##### Copyright 2024 Google LLC.

In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Gemma - 4 Multimodal Inference with vLLM.

Google released Gemma-4 its new variant that comes with Multimodality, Multilingual support upto 140 global languages, 128K context window and also Agentic capabilites like structured responses and function calling. In this notebook, we will explore how to use Multimodal inference with vLLM via Gemma 4 E2B-it model

<table align="left">
  <td>
    <a target="_blank" href="https://colab.research.google.com/github/sachinsshetty/cookbook/blob/gemma4_multimodal_inference_vllm/Gemma/%5BGemma_4%5DMulitmodal_inference_with_vLLM.ipynb"><img src="https://www.tensorflow.org/images/colab_logo_32px.png" />Run in Google Colab</a>
  </td>
</table>

## Setup

### Select the Colab runtime
To complete this tutorial, you'll need to have a Colab runtime with sufficient resources to run the Gemma model. In this case, you can use a L4 GPU:

1. In the upper-right of the Colab window, select **▾ (Additional connection options)**.
2. Select **Change runtime type**.
3. Under **Hardware accelerator**, select **L4 GPU**.

### Gemma-4 setup

To complete this tutorial, you'll first need to complete the setup instructions at [Gemma setup](https://ai.google.dev/gemma/docs/setup). The Gemma setup instructions show you how to do the following:

* Get access to Gemma on kaggle.com.
* Select a Colab runtime with sufficient resources to run
  the Gemma 4 E2B-it model.
* Generate and configure a Kaggle username and an API key as Colab secrets.

After you've completed the Gemma setup, move on to the next section, where you'll set environment variables for your Colab environment.


### Install dependencies

Run the cell below to install all the required dependencies.

- vLLM server
- Accelerate
- DuckDuck Go Search - for function calling

In [2]:
!UV_HTTP_TIMEOUT=300 uv pip install -U vllm[audio] --pre \
  --extra-index-url https://wheels.vllm.ai/nightly/cu130 \
  --extra-index-url https://download.pytorch.org/whl/cu130 \
  --index-strategy unsafe-best-match --extra-index-url https://pypi.nvidia.com


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.4/40.4 kB 300.3 kB/s eta 0:00:001m297.0 kB/s eta 0:00:01
  Using cached sentencepiece-0.2.1-cp312-cp312-manylinux_2_27_aarch64.manylinux_2_28_aarch64.whl.metadata (10 kB)
  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
  Using cached py_cpuinfo-9.0.0-py3-none-any.whl.metadata (794 bytes)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.0 MB/s eta 0:00:000m eta -:--:--
  Using cached tokenizers-0.22.2-cp39-abi3-manylinux_2_17_aarch64.manylinux2014_aarch64.whl.metadata (7.3 kB)
  Using cached pydantic-2.12.5-py3-none-any.whl.metadata (90 kB)
  Using cached prometheus_fastapi_instrumentator-7.1.0-py3-none-any.whl.metadata (13 kB)
  Using cached tiktoken-0.12.0-cp312-cp312-manylinux_2_28_aarch64.whl.metadata (6.7 kB)
  Using cached diskcache-5.6.3-py3-none-any.whl.metadata (20 kB)
  Using cached lark-1.2.2-py3-none-any.whl.metadata (1.8 kB)
  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
  Using ca

In [ ]:
!uv pip install transformers==5.5.0

In [5]:
!vllm serve google/gemma-4-E2B-it --max-model-len 8192

INFO 04-04 15:18:26 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.
WARNING 04-04 15:18:26 [interface.py:229] Failed to import from vllm._C: ImportError('libcudart.so.12: cannot open shared object file: No such file or directory')
ERROR 04-04 15:18:26 [config.py:29] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton'
ERROR 04-04 15:18:26 [gpt_oss_triton_kernels_moe.py:64] Failed to import Triton kernels. Please make sure your triton version is compatible. Error: No module named 'triton'
Traceback (most recent call last):
  File "/home/x1/code/external/cookbook/venv/bin/vllm", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/home/x1/code/external/cookbook/venv/lib/python3.12/site-packages/vllm/entrypoints/cli/main.py", line 68, in main
    cmd.subparser_init(subparsers).set_defaults(dispatch_function=cmd.cmd)
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^

In [ ]:
!vllm serve google/gemma-4-E2B-it  \
  --served-model-name gemma4  \
  --host 0.0.0.0  \
  --port 8000  \
  --gpu-memory-utilization 0.90  \
  --tensor-parallel-size 1  \
  --max-model-len 8192 \ 
  --max-num-seqs 16 \
  --enable-chunked-prefill \
  --enable-prefix-caching \
  --generation-config auto \
  --enforce-eager \
  --chat-template-content-format openai \
  --trust-remote-code  \
  --mm-encoder-tp-mode data \       
  --limit-mm-per-prompt image=2,video=1,audio=1 \
  --reasoning-parser gemma4 \
  --enable-auto-tool-choice \
  --tool-call-parser gemma4


# Gemma 4

Google has released Gemma-4, a successor to its earlier Gemma models, focusing on open-source accessibility and multimodal vision capabilities. The model is designed to handle complex tasks with extended context lengths, support for multiple languages, and vision capabilities, making it suitable for a wide range of applications.

Gemma-4 has gained attention for outperforming notable models like DeepSeek V3, o3-mini, and Llama 3 405B, with extended context lengths, multilingual support, and multimodal capabilities.

- Extended context length from 32K to 128K
- Available in 2B, 4B, 21B, and 31B sizes .
- Supports an extensive range of 140 global languages.
- Additional Features: Supports function calling, where the model can generate code or instructions to call external functions, and structured outputs, enabling it to produce data in specific formats like JSON or tables.

In [ ]:
import base64
from openai import OpenAI
import os
# Initialize the client pointing to your vLLM server
client = OpenAI(
    base_url="http://localhost:8000/v1", 
    api_key="token-is-ignored-by-vllm"
)

def encode_media(path):
    """Helper to encode local files to base64 if not using URLs."""
    with open(path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")



In [ ]:

text_response = client.chat.completions.create(
    model="gemma4",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What is your name ?"}
        ]
    }]
)

print(f"Text Analysis: {text_response.choices[0].message.content}")


In [ ]:


# 1. Processing Text + Image
image_response = client.chat.completions.create(
    model="gemma4",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "What is depicted in this image?"},
            {
                "type": "image_url",
                "image_url": {"url": f"data:image/jpeg;base64,{encode_media('media/image.jpg')}"}
            }
        ]
    }]
)


print(f"Image Analysis: {image_response.choices[0].message.content}")

In [ ]:

# 2. Processing Text + Audio (Transcription/Reasoning)
audio_response = client.chat.completions.create(
    model="gemma4",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Listen to this clip and provide a summary."},
            {
                "type": "input_audio",
                "input_audio": {
                    "data": encode_media("media/kannada_sample.wav"),
                    "format": "wav"
                }
            }
        ]
    }]
)


print(f"Audio Summary: {audio_response.choices[0].message.content}")

In [ ]:
# 3. Processing Text + Video
# Note: vLLM expects video via 'video_url' or as a series of frames 
# depending on your specific vLLM version's implementation of the OpenAI spec.
video_response = client.chat.completions.create(
    model="gemma4",
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": "Describe the movement in this video."},
            {
                "type": "video_url",
                "video_url": {"url": "https://example.com/sample_video.mp4"}
            }
        ]
    }]
)


print(f"Video Description: {video_response.choices[0].message.content}")



## Function Calling Gemma

Function calling is a feature that allows large language models to interact with external systems, APIs, and tools in a structured way. This feature enables LLMs to not just generate text responses but to invoke specific functions when appropriate based on user queries. At its core, function calling involves describing functions to the LLM and having it intelligently return the function to call along with necessary arguments for particular tasks. For example, an LLM can be configured to retrieve stock price data by calling a specific function when a user asks about the current price of a particular stock.

In [ ]:
from duckduckgo_search import DDGS

def search(query:str):
  """
  search results to the user query

  Args:
      query: user prompt to fetch search results
  """
  req = DDGS()
  response = req.text(query,max_results=4)
  context = ""
  for result in response:
    context += result['body']
  return context

In [ ]:
tools = [search]

In [ ]:
query = "which teams played Cricket ICC Champions Trophy 2025 finals and who won?" # this is a realtime query, this event occured on March 9th 2025

## Define prompt for function calling

Function Calling follows these steps:

- Your application sends a prompt to the LLM along with function definitions
- The LLM analyzes the prompt and decides whether to respond directly or use defined functions
- If using functions, the LLM generates structured arguments for the function call
- Your application receives the function call details and executes the actual function
- The function results are sent back to the LLM
- The LLM provides a final response incorporating the function results

In [ ]:
conversation = [
    {
        "role": "system",
        "content": [{"type": "text", "text": """
        You are an expert search assistant. Use `search` to get the most accurate details.
        At each turn, if you decide to invoke any of the function(s), it should be wrapped with ```tool_code```. The python methods described below are imported and available,
        you can only use defined methods. The generated code should be readable and efficient. The response to a method will be wrapped in ```tool_output``` use it to call more tools or generate a helpful, friendly response.
        When using a ```tool_call``` think step by step why and how it should be used.

        The following Python methods are available:
        \`\`\`python
        def search(query:str):
          "
          search results to the user query

          Args:
             query: user prompt to fetch search results
          "
        \`\`\`

        User: \{user_message\}
        """}]
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": query}
        ]
    }
]

In [ ]:
inputs = processor.apply_chat_template(
            conversation,
            tools=tools,
            add_generation_prompt=True,
            return_dict=True,
            tokenize=True,
            return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

In [ ]:
outputs = model.generate(**inputs, max_new_tokens=512)

In [ ]:
output = processor.decode(outputs[0], skip_special_tokens=True)

In [ ]:
response = output.split("model")[-1]

In [ ]:
print(response)


```tool_code
search(query="Which teams played in the Cricket ICC Champions Trophy 2025 final and who won?")
```


## Execute the function calling code

In [ ]:
def extract_tool_call(text):
    import io
    from contextlib import redirect_stdout

    pattern = r"```tool_code\s*(.*?)\s*```"
    match = re.search(pattern, text, re.DOTALL)
    if match:
        code = match.group(1).strip()
        # Capture stdout in a string buffer
        f = io.StringIO()
        with redirect_stdout(f):
            result = eval(code)
        output = f.getvalue()
        r = result if output == '' else output
        return f'```tool_output\n{r}\n```'''
    return None

In [ ]:
keyword = extract_tool_call(response)

In [ ]:
keyword

"```tool_output\nThe ninth edition of the Champions Trophy 2025 saw India being crowned as the winners on 9th March 2025 after they overcame New Zealand in the final. Several exceptional performers lit up the tournament with the bat and ball. The best of them made it to the Team of the Tournament. Here's what the side looks like: 1. Rachin Ravindra (New Zealand)The 2025 ICC Champions Trophy was the ninth edition of the ICC Champions Trophy, a quadrennial ODI cricket tournament organised by the International Cricket Council (ICC). In November 2021 as part of the 2024-2031 ICC men's hosts cycle, the ICC announced that the 2025 Champions Trophy would be played in Pakistan. [4]On 19 December 2024, following an agreement between Board of Control for ...Official ICC Champions Trophy, 2025 Cricket website - live matches, scores, news, highlights, commentary, rankings, videos and fixtures from the International Cricket Council. ... 2025. ICC announces Champions Trophy 2025 Team of the Tourname

In [ ]:
prompt = [{
        "role": "user",
        "content": [
            {"type": "text", "text": f"Results from the search:{keyword} User_query : {query}"}
        ]
    }
]

In [ ]:
final_prompt = processor.apply_chat_template(
            prompt,
            tools=tools,
            add_generation_prompt=True,
            return_dict=True,
            tokenize=True,
            return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

In [ ]:
response = model.generate(**final_prompt, max_new_tokens=512)

In [ ]:
print(processor.decode(response[0], skip_special_tokens=True))

user
Results from the search:```tool_output
The ninth edition of the Champions Trophy 2025 saw India being crowned as the winners on 9th March 2025 after they overcame New Zealand in the final. Several exceptional performers lit up the tournament with the bat and ball. The best of them made it to the Team of the Tournament. Here's what the side looks like: 1. Rachin Ravindra (New Zealand)The 2025 ICC Champions Trophy was the ninth edition of the ICC Champions Trophy, a quadrennial ODI cricket tournament organised by the International Cricket Council (ICC). In November 2021 as part of the 2024-2031 ICC men's hosts cycle, the ICC announced that the 2025 Champions Trophy would be played in Pakistan. [4]On 19 December 2024, following an agreement between Board of Control for ...Official ICC Champions Trophy, 2025 Cricket website - live matches, scores, news, highlights, commentary, rankings, videos and fixtures from the International Cricket Council. ... 2025. ICC announces Champions Troph